# Ground Truth — `gt_scores`

This notebook builds the **ground truth** for diagram evaluations from `llm-assesment-v2 - gt_scores.csv`, in two versions:

1. **Ordinal** — mean of expert scores (keeps the 1–4 scale).
2. **Binarized** — `0` when mean `< 2.5`, `1` when mean `> 2.5`, and a manual decision (`0` or `1`) when the mean is exactly `2.5`.

### Expert selection rules

1. **Mandatory completeness** — any expert who did not evaluate **all 15 guidelines** of their diagram is discarded.
2. Each diagram must ideally have **2 experts** after filtering. Diagrams with fewer experts are flagged and left out until new responses arrive.
3. When more than 2 experts evaluated the same diagram, we keep those with higher `Architecture_Analysis_Experience` (`extensive > medium > low`), based on `participant_demographics.csv`.
4. Ties within the same experience level are resolved manually in `MANUAL_TIEBREAKERS`.

## 1. Setup and data loading

In [1]:
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
CSV_DIR = NOTEBOOK_DIR.parent

GT_PATH = CSV_DIR / "llm-assesment-v2 - gt_scores.csv"
DEMO_PATH = CSV_DIR / "llm-assesment-v2 - participant_demographics.csv"

OUT_DIR = NOTEBOOK_DIR
OUT_ORDINAL = OUT_DIR / "ground_truth_ordinal.csv"
OUT_BINARY = OUT_DIR / "ground_truth_binary.csv"
OUT_SELECTED_EXPERTS = OUT_DIR / "selected_experts.csv"

assert GT_PATH.exists(), f"File not found: {GT_PATH}"
assert DEMO_PATH.exists(), f"File not found: {DEMO_PATH}"
print("OK - input files found")
print(f"  gt_scores:    {GT_PATH.name}")
print(f"  demographics: {DEMO_PATH.name}")

OK - input files found
  gt_scores:    llm-assesment-v2 - gt_scores.csv
  demographics: llm-assesment-v2 - participant_demographics.csv


In [2]:
gt_raw = pd.read_csv(GT_PATH)
demo = pd.read_csv(DEMO_PATH)

gt = gt_raw[gt_raw["Submission_ID"].notna()].copy()
gt = gt[gt["Submission_ID"] != "Submission_ID"]
gt["Human_GT_Score"] = pd.to_numeric(gt["Human_GT_Score"], errors="coerce")

print(f"Valid rows in gt_scores: {len(gt)}")
print(f"Participants in demographics: {len(demo)}")
gt.head()

Valid rows in gt_scores: 216
Participants in demographics: 16


,Submission_ID,Expert_ID,Diagram_ID,Criterion,Guideline,Human_GT_Score,Expert_Comment,Timestamp
0,figueiredo-dcc-ufmg-br-03dbecc3-aecc-4906-9880...,figueiredo@dcc.ufmg.br,research-modifiability-pattern-experiment,accuracy,component_matching,2,NaN,2026-05-25T18:30:01.455Z
1,figueiredo-dcc-ufmg-br-03dbecc3-aecc-4906-9880...,figueiredo@dcc.ufmg.br,research-modifiability-pattern-experiment,accuracy,interaction_matching,1,NaN,2026-05-25T18:30:01.455Z
2,figueiredo-dcc-ufmg-br-03dbecc3-aecc-4906-9880...,figueiredo@dcc.ufmg.br,research-modifiability-pattern-experiment,clarity,element_identification,3,NaN,2026-05-25T18:30:58.465Z
3,figueiredo-dcc-ufmg-br-03dbecc3-aecc-4906-9880...,figueiredo@dcc.ufmg.br,research-modifiability-pattern-experiment,clarity,reading_flow,2,NaN,2026-05-25T18:30:58.465Z
4,figueiredo-dcc-ufmg-br-03dbecc3-aecc-4906-9880...,figueiredo@dcc.ufmg.br,research-modifiability-pattern-experiment,clarity,symbol_label_readability,4,NaN,2026-05-25T18:30:58.465Z


## 2. Completeness validation

Each diagram has 15 guidelines (5 criteria × 3 guidelines on average). An expert who did not evaluate all guidelines of their diagram is considered **incomplete** and dropped entirely.

> ⚠️ If this filter leaves a diagram with fewer than 2 experts, it is flagged and **excluded from the ground truth**. Just re-run the notebook once new responses arrive.

In [3]:
expected_guidelines = (
    gt.groupby("Diagram_ID")
    .apply(lambda g: g[["Criterion", "Guideline"]].drop_duplicates().shape[0], include_groups=False)
    .to_dict()
)

coverage = (
    gt.groupby(["Diagram_ID", "Expert_ID"])
    .size()
    .reset_index(name="n_evaluations")
)
coverage["expected"] = coverage["Diagram_ID"].map(expected_guidelines)
coverage["is_complete"] = coverage["n_evaluations"] == coverage["expected"]

incomplete = coverage[~coverage["is_complete"]]
print(f"Incomplete experts detected: {len(incomplete)}")
if len(incomplete):
    display(incomplete)

complete_pairs = set(
    map(tuple, coverage.loc[coverage["is_complete"], ["Diagram_ID", "Expert_ID"]].values)
)
gt_complete = gt[
    gt.apply(lambda r: (r["Diagram_ID"], r["Expert_ID"]) in complete_pairs, axis=1)
].copy()
print(f"\nRows kept (only complete experts): {len(gt_complete)} of {len(gt)}")

Incomplete experts detected: 1


,Diagram_ID,Expert_ID,n_evaluations,expected,is_complete
10,sentry,a@g.com,6,15,False



Rows kept (only complete experts): 210 of 216


## 3. Experts × diagrams mapping (post completeness filter)

Lists which experts (already filtered as complete) evaluated each diagram. Diagrams with more than 2 experts go to the selection step; diagrams with fewer than 2 are flagged for exclusion.

In [4]:
experts_per_diagram = (
    gt_complete.groupby("Diagram_ID")["Expert_ID"]
    .unique()
    .apply(list)
    .reset_index(name="experts")
)
experts_per_diagram["n_experts"] = experts_per_diagram["experts"].str.len()
experts_per_diagram

,Diagram_ID,experts,n_experts
0,appwrite,"[lucassantoszatta@gmail.com, rodrigo.sil.borge...",3
1,geoserver-cloud,"[aafernandes1910@gmail.com, pedro.b.pimenta1@g...",2
2,magda-io,"[silvasouzasaymon@gmail.com, bernardolima95@gm...",3
3,research-modifiability-pattern-experiment,"[figueiredo@dcc.ufmg.br, samirtsj@gmail.com]",2
4,sentry,"[joaoluizlopesmegale@gmail.com, joaolfcerqueir...",2
5,spryker,"[gustavovale@ufla.br, garcia.pedro.clair@gmail...",2


In [5]:
EXPERIENCE_ORDER = {"extensive": 3, "medium": 2, "low": 1}

exp_lookup = (
    demo[["Expert_ID", "Architecture_Analysis_Experience"]]
    .drop_duplicates(subset=["Expert_ID"])
    .set_index("Expert_ID")["Architecture_Analysis_Experience"]
    .to_dict()
)

print("Architecture_Analysis_Experience per expert present in gt_scores:")
for diag_row in experts_per_diagram.itertuples():
    print(f"\n  {diag_row.Diagram_ID} (n={diag_row.n_experts}):")
    for e in diag_row.experts:
        level = exp_lookup.get(e, "??")
        print(f"    - {e:45s} -> {level}")

Architecture_Analysis_Experience per expert present in gt_scores:

  appwrite (n=3):
    - lucassantoszatta@gmail.com                    -> extensive
    - rodrigo.sil.borges@gmail.com                  -> medium
    - deconogueira1@gmail.com                       -> medium

  geoserver-cloud (n=2):
    - aafernandes1910@gmail.com                     -> medium
    - pedro.b.pimenta1@gmail.com                    -> medium

  magda-io (n=3):
    - silvasouzasaymon@gmail.com                    -> extensive
    - bernardolima95@gmail.com                      -> medium
    - mmdutra14@gmail.com                           -> extensive

  research-modifiability-pattern-experiment (n=2):
    - figueiredo@dcc.ufmg.br                        -> medium
    - samirtsj@gmail.com                            -> medium

  sentry (n=2):
    - joaoluizlopesmegale@gmail.com                 -> low
    - joaolfcerqueira4@gmail.com                    -> medium

  spryker (n=2):
    - gustavovale@ufla.br        

## 4. Selecting 2 experts per diagram

For each diagram with **3+ experts**, sort by `Architecture_Analysis_Experience` (extensive > medium > low) and keep the top 2.

Diagrams with **only 1 expert** after the completeness filter are flagged as **excluded** and re-enter the pipeline once new responses arrive.

**Ties** within the same level require a manual decision via `MANUAL_TIEBREAKERS` — currently configured for `appwrite` (keeping `rodrigo.sil.borges` alongside `lucassantoszatta`).

In [6]:
# Manual tiebreakers: for each Diagram_ID, list of Expert_IDs to KEEP when
# ranking by Architecture_Analysis_Experience does not resolve the tie alone.
MANUAL_TIEBREAKERS: dict[str, list[str]] = {
    "appwrite": [
        "P011",    # extensive
        "P014",  # medium (manually chosen over deconogueira)
    ],
}


def select_two_experts(diagram_id: str, expert_ids: list[str]) -> tuple[list[str], str]:
    """Returns (kept_experts, status). Status is 'ok' or 'excluded_too_few_experts'."""
    if len(expert_ids) < 2:
        return list(expert_ids), "excluded_too_few_experts"

    if diagram_id in MANUAL_TIEBREAKERS:
        chosen = MANUAL_TIEBREAKERS[diagram_id]
        assert len(chosen) == 2, f"{diagram_id}: tiebreaker must contain exactly 2 experts"
        for e in chosen:
            assert e in expert_ids, f"{diagram_id}: expert {e} is not among the complete ones"
        return chosen, "ok"

    if len(expert_ids) == 2:
        return list(expert_ids), "ok"

    ranked = sorted(
        expert_ids,
        key=lambda e: EXPERIENCE_ORDER.get(exp_lookup.get(e, ""), 0),
        reverse=True,
    )
    second_level = EXPERIENCE_ORDER.get(exp_lookup.get(ranked[1], ""), 0)
    third_level = EXPERIENCE_ORDER.get(exp_lookup.get(ranked[2], ""), 0)
    if third_level == second_level:
        raise ValueError(
            f"{diagram_id}: tie between {ranked[1]} and {ranked[2]} "
            f"(both at level {exp_lookup.get(ranked[1])}). "
            f"Set MANUAL_TIEBREAKERS['{diagram_id}']."
        )
    return ranked[:2], "ok"


selection_rows = []
for row in experts_per_diagram.itertuples():
    chosen, status = select_two_experts(row.Diagram_ID, row.experts)
    discarded = [e for e in row.experts if e not in chosen]
    selection_rows.append(
        {
            "Diagram_ID": row.Diagram_ID,
            "n_original": row.n_experts,
            "kept": chosen,
            "discarded": discarded,
            "kept_levels": [exp_lookup.get(e) for e in chosen],
            "status": status,
        }
    )

selection_df = pd.DataFrame(selection_rows)
excluded = selection_df[selection_df["status"] == "excluded_too_few_experts"]
if len(excluded):
    print("Diagrams excluded due to < 2 complete experts:")
    for r in excluded.itertuples():
        print(f"  - {r.Diagram_ID} (only {r.n_original} expert(s): {r.kept})")
    print()
selection_df

,Diagram_ID,n_original,kept,discarded,kept_levels,status
0,appwrite,3,"[lucassantoszatta@gmail.com, rodrigo.sil.borge...",[deconogueira1@gmail.com],"[extensive, medium]",ok
1,geoserver-cloud,2,"[aafernandes1910@gmail.com, pedro.b.pimenta1@g...",[],"[medium, medium]",ok
2,magda-io,3,"[silvasouzasaymon@gmail.com, mmdutra14@gmail.com]",[bernardolima95@gmail.com],"[extensive, extensive]",ok
3,research-modifiability-pattern-experiment,2,"[figueiredo@dcc.ufmg.br, samirtsj@gmail.com]",[],"[medium, medium]",ok
4,sentry,2,"[joaoluizlopesmegale@gmail.com, joaolfcerqueir...",[],"[low, medium]",ok
5,spryker,2,"[gustavovale@ufla.br, garcia.pedro.clair@gmail...",[],"[extensive, low]",ok


## 5. Dataset filtering

Keeps only the rows of the selected experts in diagrams with status `ok`.

In [7]:
allowed_pairs = set()
for row in selection_df.itertuples():
    if row.status != "ok":
        continue
    for e in row.kept:
        allowed_pairs.add((row.Diagram_ID, e))

mask = gt_complete.apply(
    lambda r: (r["Diagram_ID"], r["Expert_ID"]) in allowed_pairs, axis=1
)
gt_filtered = gt_complete[mask].copy()

check = (
    gt_filtered.groupby(["Diagram_ID", "Criterion", "Guideline"])
    .size()
    .reset_index(name="n")
)
bad = check[check["n"] != 2]
print(f"Rows after filtering: {len(gt_filtered)}")
print(f"Diagrams kept: {gt_filtered['Diagram_ID'].nunique()}")
print(f"(diagram, criterion, guideline) combinations with N != 2: {len(bad)}")
if len(bad):
    display(bad)
else:
    print("OK - every combination has exactly 2 evaluations")

Rows after filtering: 180
Diagrams kept: 6
(diagram, criterion, guideline) combinations with N != 2: 0
OK - every combination has exactly 2 evaluations


## 6. Ordinal ground truth (mean)

In [8]:
gt_ordinal = (
    gt_filtered.groupby(["Diagram_ID", "Criterion", "Guideline"], as_index=False)
    .agg(
        gt_score_mean=("Human_GT_Score", "mean"),
        gt_score_min=("Human_GT_Score", "min"),
        gt_score_max=("Human_GT_Score", "max"),
        n_experts=("Human_GT_Score", "count"),
    )
)
gt_ordinal

,Diagram_ID,Criterion,Guideline,gt_score_mean,gt_score_min,gt_score_max,n_experts
0,appwrite,accuracy,component_matching,2.5,2,3,2
1,appwrite,accuracy,interaction_matching,2.0,2,2,2
2,appwrite,clarity,element_identification,3.5,3,4,2
3,appwrite,clarity,reading_flow,2.0,1,3,2
4,appwrite,clarity,specific_symbology,1.0,1,1,2
...,...,...,...,...,...,...,...
85,spryker,consistency,notation_uniformity,3.0,3,3,2
86,spryker,consistency,technology_symbols,2.0,2,2,2
87,spryker,level_of_detail,element_detail,2.0,2,2,2
88,spryker,level_of_detail,interaction_detail,1.5,1,2,2


## 7. Binarized ground truth

The ordinal mean is collapsed onto a binary label:

- `mean < 2.5` → **0** (negative evaluation)
- `mean > 2.5` → **1** (positive evaluation)
- `mean == 2.5` → manual decision, either `0` or `1` (fill in `BOUNDARY_DECISIONS` below)

The next cell lists every boundary case found, so you can populate the dictionary. If any boundary case is left unresolved, the binarized CSV is **not** written, to avoid stale or misleading files.

In [9]:
boundary = gt_ordinal[gt_ordinal["gt_score_mean"] == 2.5].copy()

print(f"Boundary cases (mean == 2.5): {len(boundary)}\n")
if len(boundary):
    boundary_details = boundary.merge(
        gt_filtered[["Diagram_ID", "Criterion", "Guideline", "Expert_ID", "Human_GT_Score"]],
        on=["Diagram_ID", "Criterion", "Guideline"],
        how="left",
    )
    display(boundary_details.sort_values(["Diagram_ID", "Criterion", "Guideline"]))

    print("\nKeys for BOUNDARY_DECISIONS (copy into the cell below, use 0 for negative or 1 for positive):")
    for r in boundary.itertuples():
        print(f'    ("{r.Diagram_ID}", "{r.Criterion}", "{r.Guideline}"): None,  # e.g. 0 or 1')

Boundary cases (mean == 2.5): 15



,Diagram_ID,Criterion,Guideline,gt_score_mean,gt_score_min,gt_score_max,n_experts,Expert_ID,Human_GT_Score
0,appwrite,accuracy,component_matching,2.5,2,3,2,lucassantoszatta@gmail.com,2
1,appwrite,accuracy,component_matching,2.5,2,3,2,rodrigo.sil.borges@gmail.com,3
2,appwrite,completeness,main_interactions_coverage,2.5,2,3,2,lucassantoszatta@gmail.com,2
3,appwrite,completeness,main_interactions_coverage,2.5,2,3,2,rodrigo.sil.borges@gmail.com,3
4,appwrite,consistency,naming_consistency,2.5,2,3,2,lucassantoszatta@gmail.com,2
5,appwrite,consistency,naming_consistency,2.5,2,3,2,rodrigo.sil.borges@gmail.com,3
6,appwrite,consistency,technology_symbols,2.5,1,4,2,lucassantoszatta@gmail.com,4
7,appwrite,consistency,technology_symbols,2.5,1,4,2,rodrigo.sil.borges@gmail.com,1
8,appwrite,level_of_detail,overall_appropriateness,2.5,2,3,2,lucassantoszatta@gmail.com,2
9,appwrite,level_of_detail,overall_appropriateness,2.5,2,3,2,rodrigo.sil.borges@gmail.com,3



Keys for BOUNDARY_DECISIONS (copy into the cell below, use 0 for negative or 1 for positive):
    ("appwrite", "accuracy", "component_matching"): None,  # e.g. 0 or 1
    ("appwrite", "completeness", "main_interactions_coverage"): None,  # e.g. 0 or 1
    ("appwrite", "consistency", "naming_consistency"): None,  # e.g. 0 or 1
    ("appwrite", "consistency", "technology_symbols"): None,  # e.g. 0 or 1
    ("appwrite", "level_of_detail", "overall_appropriateness"): None,  # e.g. 0 or 1
    ("magda-io", "clarity", "specific_symbology"): None,  # e.g. 0 or 1
    ("magda-io", "clarity", "symbol_label_readability"): None,  # e.g. 0 or 1
    ("magda-io", "completeness", "external_actors_coverage"): None,  # e.g. 0 or 1
    ("magda-io", "consistency", "technology_symbols"): None,  # e.g. 0 or 1
    ("research-modifiability-pattern-experiment", "clarity", "element_identification"): None,  # e.g. 0 or 1
    ("sentry", "accuracy", "component_matching"): None,  # e.g. 0 or 1
    ("sentry", "level

In [10]:
LOW_LABEL = 0   # mean < 2.5
HIGH_LABEL = 1  # mean > 2.5

# FILL each key listed above with LOW_LABEL (0) or HIGH_LABEL (1).
# Any boundary case not mapped here will cause binarization to fail
# AND the binary CSV will not be (over)written.
BOUNDARY_DECISIONS: dict[tuple[str, str, str], int] = {
    # User decisions (2026-06-10)
    ("appwrite", "accuracy", "component_matching"): HIGH_LABEL,
    ("appwrite", "completeness", "main_interactions_coverage"): HIGH_LABEL,
    ("appwrite", "consistency", "naming_consistency"): LOW_LABEL,
    ("appwrite", "consistency", "technology_symbols"): LOW_LABEL,
    ("appwrite", "level_of_detail", "overall_appropriateness"): HIGH_LABEL,
    ("magda-io", "clarity", "specific_symbology"): LOW_LABEL,
    ("magda-io", "clarity", "symbol_label_readability"): HIGH_LABEL,
    ("magda-io", "completeness", "external_actors_coverage"): LOW_LABEL,
    ("magda-io", "consistency", "technology_symbols"): HIGH_LABEL,
    ("research-modifiability-pattern-experiment", "clarity", "element_identification"): HIGH_LABEL,
    ("sentry", "accuracy", "component_matching"): HIGH_LABEL,
    ("sentry", "level_of_detail", "element_detail"): HIGH_LABEL,
    ("spryker", "accuracy", "component_matching"): HIGH_LABEL,
    ("spryker", "completeness", "external_actors_coverage"): LOW_LABEL,
    ("spryker", "completeness", "main_interactions_coverage"): LOW_LABEL,
}


def binarize(row: pd.Series) -> int:
    mean = row["gt_score_mean"]
    if mean < 2.5:
        return LOW_LABEL
    if mean > 2.5:
        return HIGH_LABEL
    key = (row["Diagram_ID"], row["Criterion"], row["Guideline"])
    if key not in BOUNDARY_DECISIONS:
        raise KeyError(
            f"Boundary case without manual decision: {key}. "
            "Add it to BOUNDARY_DECISIONS."
        )
    val = BOUNDARY_DECISIONS[key]
    assert val in (LOW_LABEL, HIGH_LABEL), (
        f"BOUNDARY_DECISIONS[{key}] must be {LOW_LABEL} or {HIGH_LABEL}"
    )
    return val


binary_ok = False
gt_binary: pd.DataFrame | None = None
try:
    labels = gt_ordinal.apply(binarize, axis=1)
    gt_binary = gt_ordinal[["Diagram_ID", "Criterion", "Guideline"]].copy()
    gt_binary["gt_label"] = labels.astype(int)
    binary_ok = True
    print(f"Binarization OK - {len(gt_binary)} rows")
    print(f"Label distribution:\n{gt_binary['gt_label'].value_counts().sort_index().to_string()}")
    display(gt_binary)
except KeyError as exc:
    print("Binarization FAILED:", exc)
    print("Fill BOUNDARY_DECISIONS and re-run the notebook.")

Binarization OK - 90 rows


Label distribution:
gt_label
0    37
1    53


,Diagram_ID,Criterion,Guideline,gt_label
0,appwrite,accuracy,component_matching,1
1,appwrite,accuracy,interaction_matching,0
2,appwrite,clarity,element_identification,1
3,appwrite,clarity,reading_flow,0
4,appwrite,clarity,specific_symbology,0
...,...,...,...,...
85,spryker,consistency,notation_uniformity,1
86,spryker,consistency,technology_symbols,0
87,spryker,level_of_detail,element_detail,0
88,spryker,level_of_detail,interaction_detail,0


## 8. Export results

In [11]:
selection_export = selection_df.copy()
selection_export["kept"] = selection_export["kept"].apply(lambda xs: "; ".join(xs))
selection_export["discarded"] = selection_export["discarded"].apply(lambda xs: "; ".join(xs))
selection_export["kept_levels"] = selection_export["kept_levels"].apply(lambda xs: "; ".join(map(str, xs)))

gt_ordinal.to_csv(OUT_ORDINAL, index=False)
selection_export.to_csv(OUT_SELECTED_EXPERTS, index=False)

print("Generated files:")
print(f"  {OUT_ORDINAL.name}         ({len(gt_ordinal)} rows)")
print(f"  {OUT_SELECTED_EXPERTS.name} ({len(selection_export)} rows)")

if binary_ok and gt_binary is not None:
    gt_binary.to_csv(OUT_BINARY, index=False)
    print(f"  {OUT_BINARY.name}          ({len(gt_binary)} rows)")
else:
    if OUT_BINARY.exists():
        OUT_BINARY.unlink()
        print(f"  {OUT_BINARY.name} (removed - binarization is incomplete)")
    else:
        print(f"  {OUT_BINARY.name} (not written - binarization is incomplete)")

Generated files:
  ground_truth_ordinal.csv         (90 rows)
  selected_experts.csv (6 rows)
  ground_truth_binary.csv          (90 rows)
